In [ ]:
#@title Setup (run once, then collapse)
!pip install -q ipywidgets plotly
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
print('Ready!')

# Section 2: Classical ML for Product Managers

## The Knight Capital Story

In August 2012, Knight Capital deployed a trading algorithm update. A configuration error caused **millions of unintended trades in 45 minutes**, resulting in a **$440 million loss** — enough to bankrupt the company.

The ML model wasn't wrong. The **system around it** — deployment, monitoring, kill switches — failed.

**Lesson:** The ML model is the easy part. The product decisions around it determine success or catastrophe.

---

## Interactive Tools

| Tool | Link |
|------|------|
| Classical ML Playground | [Launch](https://huggingface.co/spaces/datatrainers/classical-ml-playground) |
| Metrics Explainer | [Launch](https://huggingface.co/spaces/datatrainers/metrics-explainer) |
| Data Drift Simulator | [Launch](https://huggingface.co/spaces/datatrainers/data-drift-simulator) |
| LLM vs ML Showdown | [Launch](https://huggingface.co/spaces/datatrainers/llm-vs-ml-showdown) |

---

## Exercise 1: Which Metric Matters?

For each scenario, select the metric you'd optimize for and explain why.

In [ ]:
#@title Exercise 1: Which Metric Matters?

scenarios = [
    {
        "title": "Fraud Detection at a Bank",
        "description": "FP cost: $50 (customer friction). FN cost: $5,000 (fraud loss).",
        "correct": "Recall",
        "explanation": "Missing fraud costs 100x more than a false alarm. Optimize for recall to catch every fraud case."
    },
    {
        "title": "Email Spam Filter",
        "description": "FP cost: $1,000+ (real email sent to spam, lost deal). FN cost: $0.10 (spam in inbox).",
        "correct": "Precision",
        "explanation": "Sending a real email to spam is catastrophic. Letting spam through is minor. Optimize for precision."
    },
    {
        "title": "Content Moderation (Children's Platform)",
        "description": "FP cost: $10 (safe content blocked, parent frustration). FN cost: $100,000+ (harmful content reaches child, legal/PR).",
        "correct": "Recall",
        "explanation": "Harmful content reaching children is unacceptable. Over-blocking is the lesser evil. Optimize for recall."
    },
    {
        "title": "Product Recommendation Engine",
        "description": "FP cost: $1 (irrelevant recommendation, minor annoyance). FN cost: $5 (missed upsell opportunity).",
        "correct": "F1 Score",
        "explanation": "Both errors have similar costs. Balance precision and recall with F1 for the best overall performance."
    }
]

score_box = widgets.Output()
answers = {}
dropdowns = []

for i, s in enumerate(scenarios):
    display(HTML(f"<h3>{i+1}. {s['title']}</h3><p>{s['description']}</p>"))
    dd = widgets.Dropdown(options=['-- Select --', 'Precision', 'Recall', 'F1 Score', 'Accuracy'], value='-- Select --', description='Metric:')
    dropdowns.append(dd)
    display(dd)

def check_answers(btn):
    score_box.clear_output()
    correct = 0
    with score_box:
        for i, (dd, s) in enumerate(zip(dropdowns, scenarios)):
            is_correct = dd.value == s['correct']
            if is_correct:
                correct += 1
            icon = '\u2705' if is_correct else '\u274c'
            display(HTML(f"<p>{icon} <b>{s['title']}</b>: {s['explanation']}</p>"))
        display(HTML(f"<h3>Score: {correct}/{len(scenarios)}</h3>"))

btn = widgets.Button(description='Check Answers', button_style='primary')
btn.on_click(check_answers)
display(btn, score_box)

## Exercise 2: Cost of Errors Calculator

Adjust the sliders to see how different precision/recall tradeoffs affect your bottom line.

In [ ]:
#@title Exercise 2: Cost of Errors Calculator
import plotly.graph_objects as go
from IPython.display import display, HTML
import ipywidgets as widgets

predictions = widgets.IntSlider(value=10000, min=1000, max=100000, step=1000, description='Predictions/mo:')
pos_rate = widgets.FloatSlider(value=5, min=0.5, max=30, step=0.5, description='Positive rate %:')
recall_s = widgets.FloatSlider(value=90, min=50, max=99, step=1, description='Recall %:')
precision_s = widgets.FloatSlider(value=80, min=50, max=99, step=1, description='Precision %:')
fp_cost_s = widgets.IntSlider(value=50, min=1, max=10000, step=10, description='FP Cost $:')
fn_cost_s = widgets.IntSlider(value=5000, min=1, max=100000, step=100, description='FN Cost $:')

output = widgets.Output()

def update(*args):
    output.clear_output(wait=True)
    preds = predictions.value
    pr = pos_rate.value / 100
    rec = recall_s.value / 100
    prec = precision_s.value / 100
    fpc = fp_cost_s.value
    fnc = fn_cost_s.value

    actual_pos = int(preds * pr)
    tp = int(actual_pos * rec)
    fn = actual_pos - tp
    fp = max(0, int(tp / prec) - tp)

    total_fp = fp * fpc
    total_fn = fn * fnc
    total = total_fp + total_fn

    fig = go.Figure(data=[
        go.Bar(name='FP Cost', x=['Monthly Error Cost'], y=[total_fp], marker_color='#f59e0b'),
        go.Bar(name='FN Cost', x=['Monthly Error Cost'], y=[total_fn], marker_color='#ef4444')
    ])
    fig.update_layout(barmode='stack', height=300, title=f'Total Monthly Error Cost: ${total:,.0f}')

    with output:
        display(HTML(f'<p><b>False Positives:</b> {fp:,} x ${fpc:,} = <b>${total_fp:,.0f}</b></p>'))
        display(HTML(f'<p><b>False Negatives:</b> {fn:,} x ${fnc:,} = <b>${total_fn:,.0f}</b></p>'))
        display(HTML(f'<h3>Annual Cost: ${total * 12:,.0f}</h3>'))
        fig.show()

for w in [predictions, pos_rate, recall_s, precision_s, fp_cost_s, fn_cost_s]:
    w.observe(update, 'value')

display(predictions, pos_rate, recall_s, precision_s, fp_cost_s, fn_cost_s, output)
update()

## Exercise 3: Credit Default Prediction — Design Success Metrics

You're the PM for a **credit scoring model** at a lending company. The model predicts whether a loan applicant will default.

**Context:**
- Average loan: $25,000
- Default rate: 8%
- Cost of a default (FN): ~$15,000 (after recovery)
- Cost of rejecting a good customer (FP): ~$2,000 (lost revenue)
- Regulatory requirement: Must explain why a loan was denied

Answer the questions below:

In [ ]:
#@title Exercise 3: Credit Default — Design Your Success Metrics

q1 = widgets.Dropdown(
    options=['-- Select --', 'Precision', 'Recall', 'F1 Score', 'Accuracy'],
    value='-- Select --',
    description='Primary metric:'
)
q2 = widgets.Dropdown(
    options=['-- Select --', 'Random Forest', 'Neural Network', 'Logistic Regression', 'LLM (GPT-4)'],
    value='-- Select --',
    description='Model choice:'
)
q3 = widgets.Dropdown(
    options=['-- Select --', 'Weekly', 'Monthly', 'Quarterly', 'Yearly'],
    value='-- Select --',
    description='Retrain freq:'
)

output3 = widgets.Output()

def check_credit(btn):
    output3.clear_output()
    with output3:
        score = 0
        # Q1: Recall is best (FN costs 7.5x more than FP)
        if q1.value == 'Recall':
            display(HTML('<p>\u2705 <b>Correct!</b> FN ($15K) costs 7.5x more than FP ($2K). Optimize for recall to catch defaults.</p>'))
            score += 1
        elif q1.value == 'F1 Score':
            display(HTML('<p>\u26a0\ufe0f <b>Reasonable</b>, but recall is better here since FN costs 7.5x more than FP.</p>'))
            score += 0.5
        else:
            display(HTML('<p>\u274c FN ($15K default) costs 7.5x more than FP ($2K lost revenue). Recall catches more defaults.</p>'))

        # Q2: Logistic Regression or Random Forest (explainability requirement)
        if q2.value in ['Logistic Regression', 'Random Forest']:
            display(HTML(f'<p>\u2705 <b>Correct!</b> {q2.value} provides the explainability regulators require for loan denials.</p>'))
            score += 1
        elif q2.value == 'Neural Network':
            display(HTML('<p>\u274c Neural networks are black boxes. Regulators require you to explain why a loan was denied.</p>'))
        else:
            display(HTML('<p>\u274c LLMs are too expensive, too slow, and can\'t explain decisions for regulatory compliance.</p>'))

        # Q3: Quarterly is ideal for credit
        if q3.value == 'Quarterly':
            display(HTML('<p>\u2705 <b>Correct!</b> Credit patterns shift with economic conditions. Quarterly retraining balances freshness with cost.</p>'))
            score += 1
        elif q3.value == 'Monthly':
            display(HTML('<p>\u26a0\ufe0f Monthly works but may be overkill unless economic conditions are volatile.</p>'))
            score += 0.5
        else:
            display(HTML(f'<p>\u274c {q3.value} is {"too frequent" if q3.value == "Weekly" else "too infrequent"} for credit models. Economic conditions shift quarterly.</p>'))

        display(HTML(f'<h3>Score: {score}/3</h3>'))

display(HTML('<h4>1. What is the primary metric to optimize?</h4>'))
display(q1)
display(HTML('<h4>2. Which model type? (Hint: regulatory requirement)</h4>'))
display(q2)
display(HTML('<h4>3. How often should you retrain?</h4>'))
display(q3)

btn3 = widgets.Button(description='Check Answers', button_style='primary')
btn3.on_click(check_credit)
display(btn3, output3)

## Discussion Prompts

1. **Your company's fraud model has 95% accuracy but only 60% recall.** What do you tell your VP? What's the business cost of that gap?

2. **Your data science team says they need 3 more months for data cleaning.** Leadership wants the model shipped now. How do you navigate this?

3. **A competitor launched an "AI-powered" feature using GPT-4 for something your team built with Random Forest.** Your CEO asks why you didn't use AI. What's your response?

---

## Key Takeaways

- **Accuracy is not enough.** Always ask: what's the cost of each type of error?
- **60% of ML project time is data work.** Budget accordingly.
- **Models degrade over time.** Plan for monitoring and retraining from day one.
- **Classical ML beats LLMs** on structured data (cheaper, faster, more accurate, explainable).
- **The PM's job** is to translate model metrics into business decisions.